In [4]:
import numpy as np
import matplotlib.pyplot as plt

In [12]:
### Validate ALPS calculation

## [식 8]
# CTI - conductivity tensor : σ = β · V diag(d_ext) V^T
def sigma_from_cti(beta: float, V: np.ndarray, d_ext: np.ndarray) -> np.ndarray:
    # V: (3,3) 직교행렬(고유벡터), d_ext: (3,) [d1_ext, d2_ext, d3_ext]
    return beta * V @ np.diag(d_ext) @ V.T

# ALPS Calculation formula : ALPS = (σ_xx_proj + σ_xx_assoc) / (σ_yy_proj + σ_zz_assoc)
def alps_from_sigma(sigma_proj: np.ndarray, sigma_assoc: np.ndarray) -> float:
    return (sigma_proj[0,0] + sigma_assoc[0,0]) / (sigma_proj[1,1] + sigma_assoc[2,2])

# β (beta) 가 달라도 ALPS가 동일함을 확인 : SD의 eigen vector (== 좌표축 정렬 : V = I)
I = np.eye(3)

## [식 (7)] 을 통해 유도된 'd_proj', 'd_assoc'
d_proj = np.array([1.20, 0.80, 0.60])   # d1_ext, d2_ext, d3_ext (proj ROI; 식[7])
d_assoc = np.array([1.10, 0.70, 0.50])  # d1_ext, d2_ext, d3_ext (assoc ROI)

## [식 8]의 β
# 서로 다른 β 대입
beta1, beta2, beta3 = 0.3, 0.5, 0.7

## [식 8] 을 통해 유도된 'σ' 계산
sig_proj_b1  = sigma_from_cti(beta1, I, d_proj)
sig_assoc_b1 = sigma_from_cti(beta1, I, d_assoc)
sig_proj_b2  = sigma_from_cti(beta2, I, d_proj)
sig_assoc_b2 = sigma_from_cti(beta2, I, d_assoc)
sig_proj_b3  = sigma_from_cti(beta3, I, d_proj)
sig_assoc_b3 = sigma_from_cti(beta3, I, d_assoc)

# ALPS calc (DTI-ALPS 공식 --> 'σ' 성분으로 치환)
alps_b1 = alps_from_sigma(sig_proj_b1, sig_assoc_b1)
alps_b2 = alps_from_sigma(sig_proj_b2, sig_assoc_b2)
alps_b3 = alps_from_sigma(sig_proj_b3, sig_assoc_b3)

## !!!! 두 값이 같으면 β는 완전히 소거될 것 !!!!
print("ALPS (β=2.0):", alps_b1)
print("ALPS (β=5.0):", alps_b2)
print("ALPS (β=10.0):", alps_b3)

ALPS (β=2.0): 1.769230769230769
ALPS (β=5.0): 1.769230769230769
ALPS (β=10.0): 1.7692307692307692


In [13]:
# Additional Validation : 축정렬이 아닐 때도 β 소거 확인

assert np.isclose(alps_b1, alps_b2) and np.isclose(alps_b2, alps_b3)
rng = np.random.default_rng(0)

# 회전행렬 'V (임의의 fibre dir/좌표계)' 생성
Q, _ = np.linalg.qr(rng.normal(size=(3,3)))
for b in (0.7, 3.0, 9.0):
    ap = alps_from_sigma(sigma_from_cti(b, Q, d_proj),
                         sigma_from_cti(b, Q, d_assoc))
    print(b, ap)

0.7 0.61736471757282
3.0 0.6173647175728199
9.0 0.6173647175728199


In [2]:
import sympy as sp

# ---- Scalars (shared) ----
beta = sp.symbols('beta', nonzero=True, real=True)
Dep, Dip, vip = sp.symbols('Dep Dip vip', real=True)   # projection: Dext, Dint, vint
Dea, Dia, via = sp.symbols('Dea Dia via', real=True)   # association: Dext, Dint, vint

# 식(7): extraneurite eigenvalues (아래 자리에 네 식(7)을 그대로 넣기)
f1p = sp.Function('f1p'); f2p = sp.Function('f2p'); f3p = sp.Function('f3p')
f1a = sp.Function('f1a'); f2a = sp.Function('f2a'); f3a = sp.Function('f3a')
d1p = f1p(Dep, Dip, vip); d2p = f2p(Dep, Dip, vip); d3p = f3p(Dep, Dip, vip)
d1a = f1a(Dea, Dia, via); d2a = f2a(Dea, Dia, via); d3a = f3a(Dea, Dia, via)

# S_D (= 회전/방향코사인 행렬): projection, association 각각
Sdp11,Sdp12,Sdp13,Sdp21,Sdp22,Sdp23,Sdp31,Sdp32,Sdp33 = sp.symbols('Sdp11:14 Sdp21:24 Sdp31:34', real=True)
Sda11,Sda12,Sda13,Sda21,Sda22,Sda23,Sda31,Sda32,Sda33 = sp.symbols('Sda11:14 Sda21:24 Sda31:34', real=True)
Sdp = sp.Matrix([[Sdp11,Sdp12,Sdp13],[Sdp21,Sdp22,Sdp23],[Sdp31,Sdp32,Sdp33]])
Sda = sp.Matrix([[Sda11,Sda12,Sda13],[Sda21,Sda22,Sda23],[Sda31,Sda32,Sda33]])

Dp_ext = sp.diag(d1p, d2p, d3p)
Da_ext = sp.diag(d1a, d2a, d3a)

# σ = β S_D diag(d_ext) S_D^T
sigma_p = beta * Sdp * Dp_ext * Sdp.T
sigma_a = beta * Sda * Da_ext * Sda.T

# ALPS = (σ_xx^p + σ_xx^a) / (σ_yy^p + σ_zz^a)
ALPS = sp.simplify( (sigma_p[0,0] + sigma_a[0,0]) / (sigma_p[1,1] + sigma_a[2,2]) )
ALPS = sp.simplify(sp.cancel(ALPS))  # β 약분

# 축정렬 특수형: S_D = I (첨부의 eigenvector = 표준기저 가정)
ALPS_axis = sp.simplify(ALPS.subs({
    Sdp11:1,Sdp12:0,Sdp13:0
    , Sdp21:0,Sdp22:1,Sdp23:0
    , Sdp31:0,Sdp32:0,Sdp33:1
    ,
    Sda11:1,Sda12:0,Sda13:0
    , Sda21:0,Sda22:0,Sda23:1
    , Sda31:0,Sda32:1,Sda33:0   # (x, z row만 쓰이므로 어떤 합치기로 둘지 선택)
}).subs({
    # 위 치환에서 association의 z-row를 [0,0,1]로 두고 싶다면 다음처럼:
    Sda21:0,Sda22:0,Sda23:0, Sda31:0,Sda32:0,Sda33:1
}))

In [3]:
sp.pprint(ALPS)       # β는 이미 소거

     2                           2                           2                           2                           2 ↪
Sda₁₁ ⋅f1a(Dea, Dia, via) + Sda₁₂ ⋅f2a(Dea, Dia, via) + Sda₁₃ ⋅f3a(Dea, Dia, via) + Sdp₁₁ ⋅f1p(Dep, Dip, vip) + Sdp₁₂  ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
     2                           2                           2                           2                           2 ↪
Sda₃₁ ⋅f1a(Dea, Dia, via) + Sda₃₂ ⋅f2a(Dea, Dia, via) + Sda₃₃ ⋅f3a(Dea, Dia, via) + Sdp₂₁ ⋅f1p(Dep, Dip, vip) + Sdp₂₂  ↪

↪                            2                   
↪ ⋅f2p(Dep, Dip, vip) + Sdp₁₃ ⋅f3p(Dep, Dip, vip)
↪ ───────────────────────────────────────────────
↪                            2                   
↪ ⋅f2p(Dep, Dip, vip) + Sdp₂₃ ⋅f3p(Dep, Dip, vip)


In [4]:
sp.pprint(ALPS_axis)  # (d1a + d1p)/(d2p + d3a)

f1a(Dea, Dia, via) + f1p(Dep, Dip, vip)
───────────────────────────────────────
f2a(Dea, Dia, via) + f2p(Dep, Dip, vip)
